Đọc dữ liệu

In [1]:

# CELL 2: IMPORT CAC THU VIEN

import numpy as np  # Thu vien tinh toan so hoc
import pandas as pd  # Thu vien xu ly du lieu dang bang

from pathlib import Path  # Thu vien xu ly duong dan

from sklearn.model_selection import KFold  # Chia du lieu thanh cac fold
from sklearn.model_selection import GridSearchCV  # Tim kiem sieu tham so

from sklearn.preprocessing import OrdinalEncoder  # Ma hoa du lieu phan loai
from sklearn.compose import ColumnTransformer  # Xu ly cac nhom cot
from sklearn.pipeline import Pipeline  # Ket hop tien xu ly va mo hinh

from sklearn.ensemble import RandomForestRegressor  # Mo hinh Random Forest

from sklearn.metrics import mean_squared_error  # Tinh MSE
from sklearn.metrics import mean_absolute_error  # Tinh MAE
from sklearn.metrics import r2_score  # Tinh R2

print("Da import cac thu vien thanh cong!")

# CELL 4: DOC DU LIEU

# Duong dan khi notebook chay trong experiments/random_forest
file_path = Path("../../data/raw/Diamonds Prices2022.csv")

# Duong dan du phong khi kernel chay tu thu muc goc
if not file_path.exists():
    file_path = Path("data/raw/Diamonds Prices2022.csv")

# Kiem tra tep du lieu co ton tai khong
if not file_path.exists():
    raise FileNotFoundError("Khong tim thay file Diamonds Prices2022.csv")

# Doc CSV va dung cot dau tien lam chi so dong
diamonds = pd.read_csv(file_path, index_col=0)

# In duong dan du lieu
print("Duong dan:", file_path.resolve())

# In kich thuoc ban dau
print("Kich thuoc ban dau:", diamonds.shape)

# Dem tong so gia tri thieu
print("So gia tri thieu:", diamonds.isnull().sum().sum())

# Dem so dong trung lap
print("So dong trung lap:", diamonds.duplicated().sum())

# Hien thi 5 dong dau tien
display(diamonds.head())


Da import cac thu vien thanh cong!
Duong dan: D:\DiamondPrediction\data\raw\Diamonds Prices2022.csv
Kich thuoc ban dau: (53943, 10)
So gia tri thieu: 0
So dong trung lap: 149


,carat,cut,color,clarity,depth,table,price,x,y,z
1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Tiền xử lý dữ liệu

In [2]:

# CELL 6: TIEN XU LY DU LIEU

# Tao ban sao de khong thay doi du lieu goc
df = diamonds.copy()

# Luu so dong ban dau
initial_rows = len(df)

# Loai bo cac dong co gia tri thieu
df = df.dropna()

# Luu so dong sau khi xu ly gia tri thieu
after_missing = len(df)

# Chi giu cac dong co kich thuoc x, y, z lon hon 0
df = df[
    (df["x"] > 0) &
    (df["y"] > 0) &
    (df["z"] > 0)
]

# Luu so dong sau khi loc kich thuoc
after_dimensions = len(df)

# Xoa cac dong trung lap
df = df.drop_duplicates()

# Dat lai chi so dong
df = df.reset_index(drop=True)

# In so dong bi loai do thieu du lieu
print("Dong thieu bi loai:", initial_rows - after_missing)

# In so dong bi loai do kich thuoc khong hop le
print("Dong kich thuoc khong hop le bi loai:",
      after_missing - after_dimensions)

# In so dong trung lap bi loai
print("Dong trung lap bi loai:",
      after_dimensions - len(df))

# In so dong con lai
print("So dong sau tien xu ly:", len(df))

# Kiem tra so dong trung lap con lai
print("Dong trung lap con lai:", df.duplicated().sum())

# Hien thi 5 dong dau tien
display(df.head())


Dong thieu bi loai: 0
Dong kich thuoc khong hop le bi loai: 20
Dong trung lap bi loai: 148
So dong sau tien xu ly: 53775
Dong trung lap con lai: 0


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75



## 4. Chuẩn bị dữ liệu và K-Fold

Tách dữ liệu thành:
- X: Các thuộc tính đầu vào.
- y: Giá kim cương (price).

Sử dụng:
- Outer K-Fold: 5 fold để đánh giá mô hình.
- Inner K-Fold: 3 fold để tìm kiếm siêu tham số.

Thiết lập shuffle = True và random_state = 42.

Mỗi fold Test bên ngoài chỉ được dùng để đánh giá sau khi đã chọn siêu tham số.


In [3]:

# CELL 8: CHUAN BI DU LIEU VA K-FOLD

# Lay cac thuoc tinh dau vao, bo cot price
X = df.drop(columns=["price"])

# Lay gia kim cuong lam bien muc tieu
y = df["price"]

# Khoi tao bo chia 5-Fold ben ngoai
outer_kf = KFold(
    n_splits=5,  # Chia thanh 5 fold
    shuffle=True,  # Tron du lieu truoc khi chia
    random_state=42  # Co dinh cach chia
)

# Khoi tao bo chia 3-Fold ben trong
inner_kf = KFold(
    n_splits=3,  # Chia Train thanh 3 fold
    shuffle=True,  # Tron du lieu truoc khi chia
    random_state=42  # Co dinh cach chia
)

# In so mau sau tien xu ly
print("Tong so mau:", len(X))

# In so fold ben ngoai
print("So outer fold:", outer_kf.get_n_splits())

# In so fold ben trong
print("So inner fold:", inner_kf.get_n_splits())


Tong so mau: 53775
So outer fold: 5
So inner fold: 3



# CELL 10: TAO PIPELINE RANDOM FOREST

# Thu tu chat luong mau sac
color_order = ["J", "I", "H", "G", "F", "E", "D"]

# Thu tu do tinh khiet
clarity_order = [
    "I1", "SI2", "SI1", "VS2",
    "VS1", "VVS2", "VVS1", "IF"
]

# Thu tu chat luong giac cat
cut_order = [
    "Fair", "Good", "Very Good",
    "Premium", "Ideal"
]

# Danh sach cac cot can ma hoa
categorical_cols = ["color", "clarity", "cut"]

# Khoi tao bo ma hoa voi thu tu co dinh
encoder = OrdinalEncoder(
    categories=[
        color_order,  # Thu tu cot color
        clarity_order,  # Thu tu cot clarity
        cut_order  # Thu tu cot cut
    ]
)

# Tao bo tien xu ly cho cac cot du lieu
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", encoder, categorical_cols)  # Ma hoa 3 cot
    ],
    remainder="passthrough"  # Giu nguyen cac cot so
)

# Khoi tao Random Forest
rf_model = RandomForestRegressor(
    random_state=42,  # Co dinh tinh ngau nhien
    n_jobs=-1  # Su dung cac loi CPU kha dung
)

# Tao Pipeline gom tien xu ly va mo hinh
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),  # Buoc ma hoa
        ("model", rf_model)  # Buoc huan luyen
    ]
)

# Thong bao tao Pipeline thanh cong
print("Da tao Pipeline thanh cong!")


In [4]:

# CELL 10: TAO PIPELINE RANDOM FOREST

# Thu tu chat luong mau sac
color_order = ["J", "I", "H", "G", "F", "E", "D"]

# Thu tu do tinh khiet
clarity_order = [
    "I1", "SI2", "SI1", "VS2",
    "VS1", "VVS2", "VVS1", "IF"
]

# Thu tu chat luong giac cat
cut_order = [
    "Fair", "Good", "Very Good",
    "Premium", "Ideal"
]

# Danh sach cac cot can ma hoa
categorical_cols = ["color", "clarity", "cut"]

# Khoi tao bo ma hoa voi thu tu co dinh
encoder = OrdinalEncoder(
    categories=[
        color_order,  # Thu tu cot color
        clarity_order,  # Thu tu cot clarity
        cut_order  # Thu tu cot cut
    ]
)

# Tao bo tien xu ly cho cac cot du lieu
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", encoder, categorical_cols)  # Ma hoa 3 cot
    ],
    remainder="passthrough"  # Giu nguyen cac cot so
)

# Khoi tao Random Forest
rf_model = RandomForestRegressor(
    random_state=42,  # Co dinh tinh ngau nhien
    n_jobs=-1  # Su dung cac loi CPU kha dung
)

# Tao Pipeline gom tien xu ly va mo hinh
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),  # Buoc ma hoa
        ("model", rf_model)  # Buoc huan luyen
    ]
)

# Thong bao tao Pipeline thanh cong
print("Da tao Pipeline thanh cong!")


Da tao Pipeline thanh cong!



## 6. Phạm vi tìm kiếm siêu tham số

Sử dụng cùng phạm vi tìm kiếm như mức 3 Hold-out:

- n_estimators: [100, 150]
- max_depth: [None, 20]
- min_samples_leaf: [1, 2]

Tổng cộng 8 cấu hình.

Trong mỗi outer fold, GridSearchCV sử dụng 3-Fold trên Train để lựa chọn cấu hình có RMSE trung bình thấp nhất.


In [5]:

# CELL 12: KHAI BAO PHAM VI TIM KIEM

# Khai bao danh sach cac gia tri sieu tham so
param_grid = {
    "model__n_estimators": [100, 150],  # So cay
    "model__max_depth": [None, 20],  # Do sau toi da
    "model__min_samples_leaf": [1, 2]  # So mau toi thieu tai nut la
}

# Tinh so cau hinh can thu
number_of_configs = (
    len(param_grid["model__n_estimators"])
    * len(param_grid["model__max_depth"])
    * len(param_grid["model__min_samples_leaf"])
)

# In so cau hinh
print("So cau hinh:", number_of_configs)

# In so luot huan luyen tim kiem du kien
print("So luot CV:", number_of_configs * 3 * 5)


So cau hinh: 8
So luot CV: 120



## 7. Huấn luyện Nested Cross-validation

Với mỗi outer fold:

1. Tách Train và Test.
2. Sử dụng GridSearchCV trên Train với 3-Fold.
3. Lựa chọn siêu tham số tốt nhất theo RMSE CV.
4. Huấn luyện lại mô hình tốt nhất trên toàn bộ Train.
5. Dự đoán trên Test của outer fold.
6. Lưu MSE, RMSE, MAE, R² và siêu tham số đã chọn.

Quy trình được lặp lại 5 lần.


In [6]:

# CELL 14: HUAN LUYEN NESTED K-FOLD
import time  # Thu vien tinh thoi gian
# Tao danh sach rong de luu ket qua tung fold
results = []

# Lap qua 5 fold ben ngoai
for fold, (train_idx, test_idx) in enumerate(
    outer_kf.split(X), start=1
):

    # Lay du lieu Train cua outer fold
    X_train = X.iloc[train_idx].copy()

    # Lay du lieu Test cua outer fold
    X_test = X.iloc[test_idx].copy()

    # Lay gia thuc te cua Train
    y_train = y.iloc[train_idx]

    # Lay gia thuc te cua Test
    y_test = y.iloc[test_idx]

    # Khoi tao GridSearchCV cho outer fold hien tai
    grid_search = GridSearchCV(
        estimator=pipeline,  # Pipeline Random Forest
        param_grid=param_grid,  # Pham vi tim kiem
        scoring="neg_root_mean_squared_error",  # Chon theo RMSE
        cv=inner_kf,  # Su dung 3-Fold ben trong
        n_jobs=1,  # Tranh song song long nhau
        refit=True,  # Huan luyen lai cau hinh tot nhat
        verbose=0  # Khong in chi tiet tung lan fit
    )

    # Tim kiem sieu tham so tren Train cua outer fold
    # Bắt đầu đo thời gian GridSearchCV của Fold
    fit_start = time.perf_counter()

# Thực hiện tìm kiếm siêu tham số
    grid_search.fit(X_train, y_train)

# Chỉ tính thời gian của lệnh fit
    fit_time = time.perf_counter() - fit_start

    print(f"Thời gian GridSearchCV Fold {fold}: {fit_time:.2f} giây")

    # Lay mo hinh tot nhat da duoc huan luyen
    best_model = grid_search.best_estimator_

    # Du doan tren Test cua outer fold
    y_pred = best_model.predict(X_test)

    # Tinh MSE
    mse = mean_squared_error(y_test, y_pred)

    # Tinh RMSE
    rmse = np.sqrt(mse)

    # Tinh MAE
    mae = mean_absolute_error(y_test, y_pred)

    # Tinh R2
    r2 = r2_score(y_test, y_pred)

    # Lay sieu tham so tot nhat
    best_params = grid_search.best_params_

    # Luu ket qua cua outer fold
    results.append({
        "Fold": fold,  # So thu tu fold
        "MSE": mse,  # MSE tren outer Test
        "RMSE": rmse,  # RMSE tren outer Test
        "MAE": mae,  # MAE tren outer Test
        "R2": r2,  # R2 tren outer Test
        "RMSE_CV": -grid_search.best_score_,  # RMSE inner CV
        "n_estimators": best_params["model__n_estimators"],
        "max_depth": best_params["model__max_depth"],
        "min_samples_leaf": best_params["model__min_samples_leaf"]
    })

    # In thong bao hoan thanh fold
    print(f"Fold {fold} hoan thanh")

    # In RMSE tren outer Test
    print(f"  RMSE Test: {rmse:.4f}")

    # In RMSE trung binh inner CV
    print(f"  RMSE CV: {-grid_search.best_score_:.4f}")

    # In sieu tham so tot nhat
    print(f"  Best params: {best_params}")

# Chuyen danh sach ket qua thanh DataFrame
results_df = pd.DataFrame(results)

# Hien thi ket qua cua 5 outer fold
display(results_df.round(4))


Thời gian GridSearchCV Fold 1: 77.25 giây
Fold 1 hoan thanh
  RMSE Test: 521.9015
  RMSE CV: 552.2980
  Best params: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 150}
Thời gian GridSearchCV Fold 2: 80.46 giây
Fold 2 hoan thanh
  RMSE Test: 532.5679
  RMSE CV: 552.8292
  Best params: {'model__max_depth': 20, 'model__min_samples_leaf': 2, 'model__n_estimators': 150}
Thời gian GridSearchCV Fold 3: 80.36 giây
Fold 3 hoan thanh
  RMSE Test: 534.6824
  RMSE CV: 550.2459
  Best params: {'model__max_depth': 20, 'model__min_samples_leaf': 2, 'model__n_estimators': 150}
Thời gian GridSearchCV Fold 4: 81.74 giây
Fold 4 hoan thanh
  RMSE Test: 535.0690
  RMSE CV: 555.8896
  Best params: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 150}
Thời gian GridSearchCV Fold 5: 77.98 giây
Fold 5 hoan thanh
  RMSE Test: 566.3776
  RMSE CV: 550.7419
  Best params: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators':

,Fold,MSE,RMSE,MAE,R2,RMSE_CV,n_estimators,max_depth,min_samples_leaf
0,1,272381.1712,521.9015,264.7916,0.9828,552.2980,150,NaN,2
1,2,283628.5256,532.5679,263.3510,0.9818,552.8292,150,20.0,2
2,3,285885.2844,534.6824,261.6616,0.9820,550.2459,150,20.0,2
3,4,286298.8281,535.0690,267.0259,0.9824,555.8896,150,NaN,2
4,5,320783.5654,566.3776,268.2585,0.9799,550.7419,150,NaN,2


Tổng hợp kết quả

In [7]:

# CELL 16: TONG HOP KET QUA NESTED K-FOLD

# Danh sach cac chi so can tong hop
metrics = ["MSE", "RMSE", "MAE", "R2"]

# Tinh trung binh cua tung chi so
mean_scores = results_df[metrics].mean()

# Tinh do lech chuan mau cua tung chi so
std_scores = results_df[metrics].std()

# Tao bang tong hop
summary = pd.DataFrame({
    "Chi so": metrics,  # Ten chi so
    "Trung binh": mean_scores.values,  # Gia tri trung binh
    "Do lech chuan": std_scores.values  # Do lech chuan
})

# Hien thi bang tong hop
display(summary.round(4))

# In ten thuc nghiem
print("MUC 3 - RANDOM FOREST NESTED 5-FOLD")

# In duong phan cach
print("-" * 45)

# In MSE trung binh
print(f"MSE trung binh : {mean_scores['MSE']:.4f}")

# In RMSE trung binh
print(f"RMSE trung binh: {mean_scores['RMSE']:.4f}")

# In do lech chuan RMSE
print(f"RMSE do lech chuan: {std_scores['RMSE']:.4f}")

# In MAE trung binh
print(f"MAE trung binh : {mean_scores['MAE']:.4f}")

# In R2 trung binh
print(f"R2 trung binh  : {mean_scores['R2']:.4f}")


,Chi so,Trung binh,Do lech chuan
0,MSE,289795.4749,18228.1864
1,RMSE,538.1197,16.6844
2,MAE,265.0177,2.6753
3,R2,0.9818,0.0011


MUC 3 - RANDOM FOREST NESTED 5-FOLD
---------------------------------------------
MSE trung binh : 289795.4749
RMSE trung binh: 538.1197
RMSE do lech chuan: 16.6844
MAE trung binh : 265.0177
R2 trung binh  : 0.9818


o sánh kết quả

In [8]:

# CELL 18: SO SANH MUC 2 VA MUC 3 K-FOLD

# Ket qua K-Fold muc 2 da chay
level2_rmse = 541.2329  # RMSE trung binh muc 2

level2_mae = 267.0808  # MAE trung binh muc 2

level2_r2 = 0.9815  # R2 trung binh muc 2

# Tao bang so sanh
comparison = pd.DataFrame({
    "Chi so": ["RMSE", "MAE", "R2"],
    "Muc 2": [level2_rmse, level2_mae, level2_r2],
    "Muc 3": [
        mean_scores["RMSE"],  # RMSE muc 3
        mean_scores["MAE"],  # MAE muc 3
        mean_scores["R2"]  # R2 muc 3
    ]
})

# Tinh chenh lech Muc 3 tru Muc 2
comparison["Chenh lech"] = (
    comparison["Muc 3"] - comparison["Muc 2"]
)

# Hien thi bang so sanh
display(comparison.round(4))

# Tinh phan tram thay doi RMSE
rmse_change = (
    (mean_scores["RMSE"] - level2_rmse)
    / level2_rmse * 100
)

# In phan tram thay doi
print(f"Thay doi RMSE: {rmse_change:+.2f}%")

# Kiem tra RMSE tang hay giam
if mean_scores["RMSE"] < level2_rmse:
    print("RMSE Muc 3 thap hon Muc 2.")

elif mean_scores["RMSE"] > level2_rmse:
    print("RMSE Muc 3 cao hon Muc 2.")

else:
    print("RMSE hai muc bang nhau.")


,Chi so,Muc 2,Muc 3,Chenh lech
0,RMSE,541.2329,538.1197,-3.1132
1,MAE,267.0808,265.0177,-2.0631
2,R2,0.9815,0.9818,0.0003


Thay doi RMSE: -0.58%
RMSE Muc 3 thap hon Muc 2.
